# Engineering Delivery Intelligence: analysis

This notebook answers the project's five questions in order.
The heavy lifting lives in `src/` and `sql/`; this notebook just shows the results.

**Before running:** `python src/load.py` (fills the database) and `python src/analysis.py` (draws the charts).

In [ ]:
# Let the notebook import our own code from src/
import sys
sys.path.insert(0, "../src")

import pandas as pd
from IPython.display import Image
from sqlalchemy import create_engine

from config import database_url
from analysis import cycle_time_model, yearly_trend
from feature_engineering import load_features

engine = create_engine(database_url())

def sql(query):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql(query, engine)

df = load_features()
print(f"{len(df):,} merged pull requests from {df['repo'].nunique()} projects")

## 0. Can we trust the data?
All the must-pass checks should say PASS. WARN rows are known gaps explained in `data/README.md`.

In [ ]:
sql("SELECT * FROM delivery.data_quality_report")

## 1. Where are the biggest bottlenecks?
Median days a pull request spends in each stage, per project.

In [ ]:
stages = sql("SELECT * FROM delivery.v_stage_breakdown")
stages.pivot(index="repo", columns="stage", values="median_days")

In [ ]:
Image("../reports/figures/01_where_time_goes.png")

**Finding:** in 11 of the 12 projects the longest stage is *waiting to ship*. Overall, a merged fix waits a median of about 46 days for a release, compared with about 2.6 days in review. Jaeger is the exception: its issues wait about 167 days before anyone opens a PR.

## 2. How quickly is software delivered?
The four DORA measures for each project.

In [ ]:
sql("SELECT * FROM delivery.v_dora_summary ORDER BY median_lead_time_days")

## 3. Is it getting faster without becoming less reliable?

In [ ]:
sql("SELECT * FROM delivery.v_speed_vs_reliability_yearly")

In [ ]:
# The same trend for Kubernetes alone.
# If it matches, the trend is real and not just a change in which projects were busy.
yearly_trend(df, "kubernetes/kubernetes")

In [ ]:
Image("../reports/figures/03_speed_vs_reliability.png")

**Finding:** it got slower but more reliable. Median time to merge went from 1.0 day in 2015 to 3.8 days in 2018 (Kubernetes alone: 1.2 to 6.0 days), while the share of releases needing a hotfix fell from 38% to 7%.

## 4. What is linked to slow or failed changes?

In [ ]:
sql("SELECT * FROM delivery.v_size_summary ORDER BY size_group")

In [ ]:
sql("SELECT * FROM delivery.v_review_summary ORDER BY review_group")

In [ ]:
# Regression: compares PRs within the same project.
# pct_change = how much longer a PR takes to merge when this factor goes up.
cycle_time_model(df)

In [ ]:
Image("../reports/figures/04_cycle_time_drivers.png")

**Finding:** the amount of back-and-forth matters more than size. Each doubling of review comments goes with about 52% longer to merge, and each doubling of commits about 40%. PRs opened at the weekend take about 60% longer, and fixes for issues raised by outsiders about 40% longer. PRs with more than 10 review comments are also reverted almost four times as often as PRs with none (1.5% vs 0.4%). These are links, not proof of cause: a hard change attracts both discussion and risk.

## 5. How do the projects compare?

In [ ]:
Image("../reports/figures/05_projects_speed_vs_failure.png")

Recommendations and limitations are in the README.